# Sesión 04 — Calidad, metadatos y privacidad de datos

**Bloque 6 · Gobierno del Dato y Estrategia**  
Ciclo 1 — Data Analytics

---

En la sesión anterior construimos el marco de trabajo del gobierno del dato: marcos de referencia (DAMA-DMBOK, DGI), objetivos estratégicos, la matriz RACI y un catálogo de políticas. Hoy descendemos un nivel y abordamos tres pilares operativos que hacen que ese marco funcione en la práctica:

1. **Calidad de datos** — cómo medir si los datos son fiables y útiles.
2. **Metadatos** — cómo documentar y rastrear los datos a lo largo de su ciclo de vida.
3. **Privacidad** — cómo proteger los datos personales conforme al RGPD y la LOPD-GDD.

Seguimos trabajando con el caso de **DataMart España**, la cadena minorista con 120 tiendas físicas y un e-commerce en plena digitalización.

---

## Bloque 1 — Calidad de datos (~20 min)

### ¿Qué entendemos por calidad de datos?

La calidad de datos no es un concepto binario (bueno/malo), sino un conjunto de **dimensiones medibles** que determinan si un dato es apto para el uso que se le quiere dar. Un dato puede ser técnicamente correcto pero inútil si llega tarde; puede estar completo pero ser inconsistente con otras fuentes.

El marco más extendido para evaluar la calidad define **seis dimensiones**:

| Dimensión | Pregunta que responde | Ejemplo en DataMart España |
|---|---|---|
| **Exactitud** (Accuracy) | ¿El valor refleja la realidad? | ¿La dirección del cliente es correcta? |
| **Completitud** (Completeness) | ¿Están todos los valores necesarios? | ¿Faltan códigos postales en la tabla de clientes? |
| **Consistencia** (Consistency) | ¿Los datos coinciden entre sistemas? | ¿El nombre del cliente es igual en CRM y en facturación? |
| **Unicidad** (Uniqueness) | ¿Hay registros duplicados? | ¿Un mismo cliente aparece dos veces con IDs diferentes? |
| **Validez** (Validity) | ¿Los valores cumplen reglas de formato/dominio? | ¿Los emails tienen formato correcto? ¿Las edades son positivas? |
| **Oportunidad** (Timeliness) | ¿Los datos están disponibles cuando se necesitan? | ¿El stock se actualiza antes de que abra la tienda? |

Estas dimensiones no son independientes: un dato incompleto no puede ser exacto, y un dato duplicado erosiona la unicidad y la consistencia simultáneamente. El objetivo de un programa de calidad de datos es definir **umbrales aceptables** para cada dimensión y monitorizar su cumplimiento de forma continua.

### Scorecard de calidad: midiendo las dimensiones

Vamos a construir un dataset sintético de clientes de DataMart España con problemas de calidad intencionados, y después calcularemos un «scorecard» que puntúe cada dimensión. Este tipo de cuadro de mando es lo que un equipo de gobierno del dato revisaría periódicamente para detectar degradaciones.

In [1]:
import pandas as pd
import numpy as np
import hashlib

# ── Dataset sintético: tabla de clientes de DataMart España ──
# Introducimos problemas de calidad a propósito para cada dimensión.

np.random.seed(42)

clientes = pd.DataFrame({
    "id_cliente":   [101, 102, 103, 104, 105, 106, 107, 108, 108, 110],  # 108 duplicado
    "nombre":       ["Ana García", "Pedro López", "María Ruiz", None, "Luis Sánchez",
                     "Carmen Díaz", "ana garcía", "Jorge Martín", "Jorge Martín", "Elena Torres"],
    "email":        ["ana@email.com", "pedro@email", "maria@email.com", "sofia@email.com",
                     None, "carmen@email.com", "ana@email.com", "jorge@email.com", "jorge@email.com", "elena@email.com"],
    "edad":         [34, 28, -5, 45, 52, 200, 41, 38, 38, 29],           # -5 y 200 inválidos
    "cp":           ["28001", "08002", None, "28003", "41004", "28005", "28001", "30006", "30006", "46007"],
    "ciudad":       ["Madrid", "Barcelona", "Madrid", "Madrid", "Sevilla",
                     "Madrid", "madrid", "Murcia", "Murcia", "Valencia"],  # "madrid" inconsistente
    "fecha_alta":   pd.to_datetime(["2023-01-15", "2023-03-20", "2023-02-10", "2023-04-05",
                                     "2023-05-12", "2023-06-30", "2023-01-15", "2023-07-18",
                                     "2023-07-18", "2023-08-22"]),
    "consentimiento_marketing": [True, True, False, True, None, True, True, False, False, True]
})

print(f"Registros: {len(clientes)}")
print(f"Columnas:  {list(clientes.columns)}")
clientes

Registros: 10
Columnas:  ['id_cliente', 'nombre', 'email', 'edad', 'cp', 'ciudad', 'fecha_alta', 'consentimiento_marketing']


,id_cliente,nombre,email,edad,cp,ciudad,fecha_alta,consentimiento_marketing
0,101,Ana García,ana@email.com,34,28001,Madrid,2023-01-15,True
1,102,Pedro López,pedro@email,28,08002,Barcelona,2023-03-20,True
2,103,María Ruiz,maria@email.com,-5,None,Madrid,2023-02-10,False
3,104,None,sofia@email.com,45,28003,Madrid,2023-04-05,True
4,105,Luis Sánchez,None,52,41004,Sevilla,2023-05-12,None
5,106,Carmen Díaz,carmen@email.com,200,28005,Madrid,2023-06-30,True
6,107,ana garcía,ana@email.com,41,28001,madrid,2023-01-15,True
7,108,Jorge Martín,jorge@email.com,38,30006,Murcia,2023-07-18,False
8,108,Jorge Martín,jorge@email.com,38,30006,Murcia,2023-07-18,False
9,110,Elena Torres,elena@email.com,29,46007,Valencia,2023-08-22,True


A simple vista ya podemos intuir problemas: hay un `None` en el nombre, un email sin dominio válido, edades imposibles, un código postal ausente, y filas que parecen duplicadas. Ahora sistematicemos esa intuición con métricas.

In [2]:
# ── Scorecard de calidad por dimensión ──
# Cada métrica devuelve un porcentaje (100% = perfecto).

import re

def calcular_scorecard(df):
    n = len(df)
    
    # 1. Completitud: % de celdas no nulas sobre el total
    completitud = (1 - df.isnull().sum().sum() / df.size) * 100
    
    # 2. Unicidad: % de filas únicas por clave primaria (id_cliente)
    unicidad = (df["id_cliente"].nunique() / n) * 100
    
    # 3. Validez — reglas de dominio
    email_valido = df["email"].dropna().apply(lambda x: bool(re.match(r"^[^@]+@[^@]+\.[^@]+$", x)))
    edad_valida  = df["edad"].dropna().between(0, 120)
    validez = (email_valido.sum() + edad_valida.sum()) / (len(email_valido) + len(edad_valida)) * 100
    
    # 4. Consistencia: comprobamos si las ciudades usan capitalización uniforme
    ciudades = df["ciudad"].dropna()
    consistencia = (ciudades.apply(lambda x: x == x.title()).sum() / len(ciudades)) * 100
    
    # 5. Exactitud: difícil de medir sin fuente de verdad. Proxy: emails y edades válidos
    exactitud = validez  # En la práctica requiere cruce con fuente externa
    
    # 6. Oportunidad: proxy — % de registros con fecha_alta en los últimos 12 meses
    corte = pd.Timestamp("2023-09-01")
    oportunidad = (df["fecha_alta"].dropna() >= corte - pd.DateOffset(months=12)).sum() / n * 100
    
    return pd.DataFrame({
        "Dimensión": ["Completitud", "Unicidad", "Validez", "Consistencia", "Exactitud (proxy)", "Oportunidad"],
        "Puntuación (%)": [completitud, unicidad, validez, consistencia, exactitud, oportunidad],
        "Umbral mínimo (%)": [95, 99, 98, 100, 95, 90]
    })

scorecard = calcular_scorecard(clientes)

# Coloreamos: verde si cumple umbral, rojo si no
def colorear(row):
    color = "background-color: #d4edda" if row["Puntuación (%)"] >= row["Umbral mínimo (%)"] else "background-color: #f8d7da"
    return [color] * len(row)

scorecard.style.apply(colorear, axis=1).format({"Puntuación (%)": "{:.1f}", "Umbral mínimo (%)": "{:.0f}"})

AttributeError: The '.style' accessor requires jinja2

### Reglas de validación y SLAs de calidad

El scorecard nos da una foto global, pero en la práctica el equipo de gobierno del dato necesita **reglas concretas** (business rules) vinculadas a cada campo. Estas reglas se documentan, se automatizan, y se revisan periódicamente. Cuando un campo no cumple su regla, se genera una alerta o se impide que el dato pase a la capa Gold del data lakehouse.

También es habitual pactar **SLAs (Service Level Agreements) de calidad** con los equipos consumidores: «la tabla de clientes tendrá como máximo un 2% de emails inválidos y se actualizará antes de las 8:00 cada día laboral».

In [3]:
# ── Reglas de validación por campo ──
# Cada regla es una función que devuelve True/False por fila.

reglas = {
    "email_formato":    {"campo": "email",    "descripción": "Email con formato usuario@dominio.ext",
                         "regla": lambda s: s.apply(lambda x: bool(re.match(r"^[^@]+@[^@]+\.[^@]+$", str(x))) if pd.notna(x) else False)},
    "edad_rango":       {"campo": "edad",     "descripción": "Edad entre 0 y 120",
                         "regla": lambda s: s.between(0, 120)},
    "cp_no_nulo":       {"campo": "cp",       "descripción": "Código postal presente",
                         "regla": lambda s: s.notna()},
    "nombre_no_nulo":   {"campo": "nombre",   "descripción": "Nombre presente",
                         "regla": lambda s: s.notna()},
    "ciudad_formato":   {"campo": "ciudad",   "descripción": "Ciudad en formato Title Case",
                         "regla": lambda s: s.dropna().apply(lambda x: x == x.title()).reindex(s.index, fill_value=False)},
}

# Aplicamos todas las reglas y generamos un informe
informe = []
for nombre_regla, config in reglas.items():
    serie = clientes[config["campo"]]
    resultado = config["regla"](serie)
    cumple = resultado.sum()
    total = len(serie)
    informe.append({
        "Regla": nombre_regla,
        "Campo": config["campo"],
        "Descripción": config["descripción"],
        "Cumple": cumple,
        "Total": total,
        "% Cumplimiento": round(cumple / total * 100, 1)
    })

informe_df = pd.DataFrame(informe)

def color_cumplimiento(val):
    if val >= 95:
        return "background-color: #d4edda"
    elif val >= 80:
        return "background-color: #fff3cd"
    else:
        return "background-color: #f8d7da"

informe_df.style.applymap(color_cumplimiento, subset=["% Cumplimiento"])

AttributeError: The '.style' accessor requires jinja2

---

## Bloque 2 — Metadatos (~20 min)

### La columna vertebral del gobierno del dato

Si la calidad mide «qué tan buenos son los datos», los metadatos responden a «qué son, de dónde vienen y cómo se usan». Sin metadatos, un campo llamado `mnt_bruto` en una tabla es un misterio; con metadatos, sabemos que es el «monto bruto de la transacción en euros, antes de impuestos, calculado por el sistema ERP».

Distinguimos tres tipos fundamentales:

**Metadatos técnicos** — describen la estructura de los datos tal como los ve un sistema informático: nombre de columna, tipo de dato (int, varchar, datetime), restricciones (NOT NULL, UNIQUE), longitud máxima, tabla y esquema donde reside.

**Metadatos de negocio** — describen el significado de los datos para las personas: definición en lenguaje natural, propietario del dato (Data Owner), reglas de negocio asociadas, nivel de confidencialidad.

**Metadatos operacionales** — describen el ciclo de vida del dato: cuándo se creó, cuándo se actualizó por última vez, qué proceso ETL lo alimenta, con qué frecuencia se refresca, cuántas filas tiene.

Dos herramientas clave materializan los metadatos:

- **Diccionario de datos**: documento que describe campo por campo cada tabla. Es el contrato entre quien produce el dato y quien lo consume.
- **Catálogo de datos**: herramienta (a menudo una plataforma como Apache Atlas, Alation o Azure Purview) que permite buscar, descubrir y entender los datasets disponibles en la organización. El diccionario de datos es un componente del catálogo.

### Construyendo un diccionario de datos automatizado

Una buena práctica es generar la parte técnica del diccionario de datos de forma automática a partir del propio DataFrame, y completar manualmente la parte de negocio. Esto asegura que la documentación técnica nunca queda desactualizada respecto al dato real.

In [4]:
# ── Diccionario de datos automatizado ──
# Parte técnica: se extrae del DataFrame.
# Parte de negocio: se añade manualmente (aquí simulada).

descripciones_negocio = {
    "id_cliente":    "Identificador único del cliente en el sistema CRM de DataMart España.",
    "nombre":        "Nombre completo del cliente (nombre + primer apellido).",
    "email":         "Dirección de correo electrónico principal para comunicaciones.",
    "edad":          "Edad del cliente en años, calculada a partir de la fecha de nacimiento.",
    "cp":            "Código postal del domicilio habitual del cliente.",
    "ciudad":        "Ciudad de residencia del cliente, según registro en CRM.",
    "fecha_alta":    "Fecha en que el cliente se registró por primera vez en cualquier canal.",
    "consentimiento_marketing": "Indica si el cliente ha dado consentimiento explícito para recibir comunicaciones comerciales (RGPD art. 7)."
}

propietarios = {
    "id_cliente": "CRM", "nombre": "CRM", "email": "CRM",
    "edad": "CRM", "cp": "CRM", "ciudad": "CRM",
    "fecha_alta": "CRM", "consentimiento_marketing": "Legal / DPO"
}

diccionario = []
for col in clientes.columns:
    diccionario.append({
        "Campo": col,
        "Tipo Python": str(clientes[col].dtype),
        "Nulos": clientes[col].isnull().sum(),
        "% Nulos": round(clientes[col].isnull().mean() * 100, 1),
        "Valores únicos": clientes[col].nunique(),
        "Ejemplo": str(clientes[col].dropna().iloc[0]) if clientes[col].notna().any() else "—",
        "Descripción de negocio": descripciones_negocio.get(col, "Sin documentar"),
        "Propietario": propietarios.get(col, "Sin asignar"),
    })

dicc_df = pd.DataFrame(diccionario)

# Resaltamos los campos sin documentar
def resaltar_sin_doc(val):
    return "background-color: #fff3cd" if val == "Sin documentar" else ""

dicc_df.style.applymap(resaltar_sin_doc, subset=["Descripción de negocio"])

AttributeError: The '.style' accessor requires jinja2

### Linaje de datos (Data Lineage)

El linaje responde a la pregunta: **¿de dónde viene este dato y qué transformaciones ha sufrido?** Es fundamental para:

- **Depuración**: si un KPI es incorrecto, el linaje permite rastrear el error hasta su origen.
- **Impacto**: si se modifica un campo en el sistema fuente, el linaje muestra todos los informes y dashboards afectados.
- **Cumplimiento regulatorio**: el RGPD exige saber dónde están los datos personales y cómo fluyen (artículo 30, registro de actividades de tratamiento).

El linaje se puede representar como un grafo dirigido donde los nodos son tablas o campos y las aristas son transformaciones. En organizaciones maduras, herramientas como dbt, Apache Atlas o Azure Purview generan el linaje automáticamente. En DataMart España, que está en fase temprana, lo documentamos manualmente.

Veamos el linaje del campo `email_cliente` desde su origen hasta su uso en un dashboard:

In [5]:
# ── Mini-linaje del campo email_cliente ──
# Representamos las etapas como un DataFrame tabular.

linaje_email = pd.DataFrame({
    "Etapa": ["1. Origen", "2. Ingesta (Bronze)", "3. Limpieza (Silver)", "4. Enriquecimiento (Gold)", "5. Consumo"],
    "Sistema / Capa": ["Formulario web (CRM)", "Data Lake — Bronze", "Data Lake — Silver", "Data Lake — Gold", "Dashboard Power BI"],
    "Campo": ["input_email", "raw_email", "email_limpio", "email_cliente", "email_cliente"],
    "Transformación aplicada": [
        "Entrada manual del usuario",
        "Copia cruda sin modificar (ingesta batch diaria, 02:00h)",
        "Trim + lowercase + validación regex (ETL Python)",
        "Deduplicación por id_cliente + seudonimización para analítica",
        "Lectura directa desde Gold (DirectQuery)"
    ],
    "Responsable": ["Equipo Web", "Data Engineer", "Data Engineer", "Data Steward + DPO", "Analista BI"]
})

linaje_email.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left"), ("background-color", "#4472C4"), ("color", "white")]}]
)

AttributeError: The '.style' accessor requires jinja2

---

## Bloque 3 — Privacidad y RGPD (~20 min)

### Marco legal: RGPD y LOPD-GDD

El **Reglamento General de Protección de Datos (RGPD)**, en vigor desde mayo de 2018, es la norma europea que regula el tratamiento de datos personales. En España se complementa con la **LOPD-GDD** (Ley Orgánica de Protección de Datos y Garantía de los Derechos Digitales).

Para un profesional de datos, estos son los principios que afectan directamente al trabajo diario:

| Principio | Qué implica para el equipo de datos |
|---|---|
| **Minimización** | Solo recoger y almacenar los datos estrictamente necesarios para la finalidad declarada. |
| **Limitación de finalidad** | No reutilizar datos recogidos para un fin con un propósito diferente sin nuevo consentimiento. |
| **Consentimiento** | El tratamiento requiere una base legal; si es consentimiento, debe ser libre, informado, específico e inequívoco. |
| **Derecho de acceso** | El interesado puede pedir una copia de todos sus datos personales. |
| **Derecho al olvido** | El interesado puede solicitar la supresión de sus datos (con excepciones legales). |
| **Portabilidad** | El interesado puede pedir sus datos en formato estructurado y legible por máquina. |
| **Notificación de brechas** | Obligación de notificar a la AEPD en 72 horas si hay una violación de seguridad que afecte a datos personales. |

El **DPO (Data Protection Officer / Delegado de Protección de Datos)**, cuyo rol ya vimos en la sesión 02, es la figura que vela por el cumplimiento de estas obligaciones dentro de la organización.

En DataMart España, que maneja datos de clientes (nombres, emails, direcciones, historial de compras), el RGPD tiene un impacto directo en cómo se diseña la arquitectura de datos y qué controles se implementan.

### Clasificación de datos por nivel de sensibilidad

El primer paso operativo para cumplir con la privacidad es **clasificar cada campo** según su nivel de sensibilidad. Esta clasificación determina quién puede acceder al dato, qué protecciones técnicas se aplican, y durante cuánto tiempo se conserva.

Una taxonomía habitual tiene cuatro niveles:

- **Público**: datos que se pueden compartir sin restricción (nombre de la tienda, horario comercial).
- **Interno**: datos accesibles para empleados de la organización pero no para el exterior (KPIs de ventas agregados, organigrama).
- **Confidencial**: datos cuya divulgación podría causar daño a la organización (estrategia de precios, márgenes).
- **PII (Personally Identifiable Information)**: datos que identifican o pueden identificar a una persona física (nombre, email, dirección, DNI). Estos requieren las protecciones más estrictas bajo el RGPD.

In [ ]:
# ── Clasificación de sensibilidad de cada campo ──
# En un programa de gobierno del dato real, esta clasificación la definen
# conjuntamente el Data Owner, el Data Steward y el DPO.

clasificacion = pd.DataFrame({
    "Campo": clientes.columns,
    "Nivel de sensibilidad": [
        "Interno",      # id_cliente — no identifica por sí solo, pero es clave
        "PII",          # nombre
        "PII",          # email
        "PII",          # edad
        "PII",          # cp — puede ser PII si combinado con otros datos
        "PII",          # ciudad — discutible, pero junto con cp puede identificar
        "Interno",      # fecha_alta
        "PII",          # consentimiento_marketing — dato de carácter personal
    ],
    "Base legal RGPD": [
        "Interés legítimo",
        "Ejecución de contrato",
        "Ejecución de contrato",
        "Consentimiento",
        "Ejecución de contrato",
        "Ejecución de contrato",
        "Interés legítimo",
        "Consentimiento explícito"
    ],
    "Período de retención": [
        "Mientras sea cliente + 5 años",
        "Mientras sea cliente + 5 años",
        "Mientras sea cliente + 5 años",
        "Mientras sea cliente",
        "Mientras sea cliente + 5 años",
        "Mientras sea cliente + 5 años",
        "Indefinido (dato operacional)",
        "Mientras sea cliente (revocable)"
    ]
})

# Colores por nivel
colores_nivel = {"Público": "#d4edda", "Interno": "#cce5ff", "Confidencial": "#fff3cd", "PII": "#f8d7da"}

def colorear_nivel(val):
    return f"background-color: {colores_nivel.get(val, '')}"

clasificacion.style.applymap(colorear_nivel, subset=["Nivel de sensibilidad"])

### Técnicas de protección: anonimización vs. seudonimización

Una vez clasificados los campos sensibles, el equipo de datos debe aplicar técnicas de protección antes de que los datos lleguen a entornos de analítica o se compartan con terceros. Las dos técnicas principales son:

**Anonimización** — proceso **irreversible** que elimina cualquier posibilidad de identificar a la persona. Un dato anonimizado ya no es dato personal a efectos del RGPD. Ejemplo: reemplazar el email por un valor aleatorio sin ninguna tabla de correspondencia.

**Seudonimización** — proceso **reversible** que sustituye los identificadores directos por un seudónimo (normalmente un hash). El dato sigue siendo personal (porque se puede revertir con la clave), pero reduce el riesgo en caso de brecha. El RGPD la recomienda explícitamente como medida de seguridad (artículo 32).

| | Anonimización | Seudonimización |
|---|---|---|
| ¿Reversible? | No | Sí (con la clave) |
| ¿Sigue siendo dato personal? | No | Sí |
| ¿Aplica RGPD? | No | Sí |
| Caso de uso típico | Datasets públicos, investigación | Analítica interna, entornos de test |

En DataMart España, la estrategia es:
- Capa **Gold** para analistas: datos seudonimizados (hash de email, enmascaramiento de nombre).
- Datos compartidos con partners externos: anonimización completa.
- Capa operacional (CRM): datos reales, acceso restringido.

In [ ]:
# ── Seudonimización y enmascaramiento ──
# Aplicamos dos técnicas a los campos PII del DataFrame de clientes.

def seudonimizar_hash(valor, sal="DataMartES_2024"):
    """Genera un hash SHA-256 del valor concatenado con una sal.
    La sal impide ataques de diccionario sobre hashes comunes."""
    if pd.isna(valor):
        return None
    return hashlib.sha256(f"{sal}_{valor}".encode()).hexdigest()[:16]

def enmascarar_nombre(nombre):
    """Muestra solo la inicial y enmascara el resto. Ej: 'Ana García' → 'A*** G*****'"""
    if pd.isna(nombre):
        return None
    partes = nombre.split()
    return " ".join(p[0] + "*" * (len(p) - 1) for p in partes)

def enmascarar_email(email):
    """Muestra los dos primeros caracteres y enmascara el resto antes del @."""
    if pd.isna(email):
        return None
    partes = email.split("@")
    if len(partes) != 2:
        return email
    usuario = partes[0][:2] + "***"
    return f"{usuario}@{partes[1]}"

# Creamos una versión protegida del DataFrame
clientes_protegidos = clientes.copy()
clientes_protegidos["nombre"]    = clientes["nombre"].apply(enmascarar_nombre)
clientes_protegidos["email"]     = clientes["email"].apply(enmascarar_email)
clientes_protegidos["email_hash"] = clientes["email"].apply(seudonimizar_hash)
clientes_protegidos["edad"]      = clientes["edad"].apply(lambda x: f"{(x // 10) * 10}-{(x // 10) * 10 + 9}" if pd.notna(x) and 0 <= x <= 120 else None)

print("═" * 60)
print("ANTES (datos originales — solo accesible en CRM)")
print("═" * 60)
print(clientes[["id_cliente", "nombre", "email", "edad"]].to_string(index=False))
print()
print("═" * 60)
print("DESPUÉS (datos protegidos — capa Gold para analistas)")
print("═" * 60)
print(clientes_protegidos[["id_cliente", "nombre", "email", "email_hash", "edad"]].to_string(index=False))

Observa las diferencias:

- El **nombre** se ha enmascarado parcialmente: conserva la inicial (útil para verificación) pero impide la identificación directa.
- El **email** se muestra con enmascaramiento parcial (los dos primeros caracteres), suficiente para que un analista reconozca patrones pero no identifique a la persona.
- Se ha añadido un **email_hash** (SHA-256 con sal): este valor permite vincular registros del mismo cliente entre tablas sin exponer el email real. Dos registros con el mismo email producirán el mismo hash.
- La **edad** se ha convertido en un **rango** (20-29, 30-39...), técnica de generalización que preserva la utilidad analítica (segmentación por grupo de edad) pero reduce la identificabilidad.

Estas técnicas, aplicadas de forma sistemática en el pipeline ETL, son la materialización operativa de los principios del RGPD dentro de la arquitectura de datos.

---

## Cierre y conexión con la próxima sesión

Hoy hemos recorrido los tres pilares operativos que complementan el marco de gobierno del dato:

- **Calidad**: medir, definir reglas y establecer umbrales aceptables.
- **Metadatos**: documentar qué son los datos, de dónde vienen y cómo se transforman.
- **Privacidad**: clasificar, proteger y cumplir con el marco legal.

En la **Sesión 05** (reto guiado) integraréis todo lo aprendido en el bloque: diseñaréis un **plan de gobierno del dato completo** para DataMart España, que incluya el marco de trabajo (S03), las métricas de calidad, el diccionario de datos y la estrategia de protección de datos personales (S04). Será un ejercicio que simula lo que haría un equipo real al arrancar un programa de gobierno del dato.